In [ ]:
# TGraph records carry attributes; convert to geometry only when needed.
def _as_topology(value):
    from topologicpy.Topology import Topology
    from topologicpy.Vertex import Vertex
    if isinstance(value, (list, tuple)):
        return [_as_topology(item) for item in value]
    if not isinstance(value, dict) or "index" not in value or "dictionary" not in value:
        return value
    representation = value.get("representation")
    attributes = value["dictionary"]
    if representation is None and "src" not in value:
        representation = Vertex.ByCoordinates(
            attributes.get("x", 0), attributes.get("y", 0), attributes.get("z", 0))
    if representation is None:
        raise ValueError("This graph record has no geometry representation")
    return Topology.SetDictionary(representation, attributes)


def _record_dictionary(value):
    from topologicpy.Topology import Topology
    if isinstance(value, dict) and "dictionary" in value:
        return value["dictionary"]
    return Topology.Dictionary(value)


def _edge_records(graph):
    from topologicpy.TGraph import TGraph
    records = TGraph.Edges(graph)
    if any(record.get("representation") is None for record in records):
        geometry = TGraph.Edges(graph, asTopologic=True, segmentCurves=False)
        if len(geometry) != len(records):
            raise ValueError("Could not generate one geometry representation per graph edge")
        for record, edge in zip(records, geometry):
            record["representation"] = edge
    return records


# Creating a Spatial Relationships Graph from an IFC file

In [ ]:
# This cell is not needed if you have pip installed topologicpy
import sys
sys.path.append("C:/Users/sarwj/OneDrive - Cardiff University/Documents/GitHub/topologicpy/src")

## 1. Import the needed topologicpy Classes

In [ ]:
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.TGraph import TGraph
from topologicpy.Dictionary import Dictionary
from topologicpy.Color import Color
from topologicpy.Helper import Helper

## 2. Check the TopologicPy version

In [ ]:
print("This tutorial requires topologicpy version 0.9.16 or newer.")
print(Helper.Version())

## 3. Set your renderer:
* Visual studio code: "vscode"
* Google Colab: "colab"
* Browser: "browser"

In [ ]:
renderer = "vscode"

## 4. Set default mappings (do not change key names)

In [ ]:
# Do not change the code below this line
rels_color_mapping = {"contains": "#FF0000", # Same as within
                 "coveredBy": "#0000C8", # Same as covers
                 "covers": "#0000C8", # Same as coveredBy
                 "crosses": "#0098FF",
                 "disjoint": "#2CFF96",
                 "equals": "#97FF00",
                 "overlaps": "#FFEA00",
                 "touches": "#550E55",
                 "within": "#FF0000",
                 "near": "#AAAAAA",
                 "intermediate": "#666666",
                 "far": "#000000"
                }

ifc_color_mapping = {"ifcwallstandardcase": "#FF0000", # Same as within
                 "ifcdoor": "#0000C8", # Same as covers
                 "ifcwindow": "#0000C8", # Same as coveredBy
                 "ifcslab": "#0098FF",
                 "ifcwall": "#2CFF96"
                }

## 5. Specify the path to the IFC file and what to include

In [ ]:
# Choose your own IFC file if you wish
ifc_file_path = r"C:\Users\sarwj\OneDrive - Cardiff University\Documents\GitHub\topologicpy\assets\Ifc2x3_Duplex_Architecture.ifc"
includeTypes=["ifcwall", "ifcwallstandardcase", "ifcslab", "ifcwindow", "ifcdoor"]

## 6. Specify which spatial relationships you are interested in

In [ ]:
includeRels = ["overlaps", "touches","within"]

## 7. Import the IFC file

In [ ]:
# Here you can limit the IFC types you wish to include or exclude. Read the API documentation on Topology.ByIFCPath
# https://topologicpy.readthedocs.io/en/latest/
ifc_objects = Topology.ByIFCPath(ifc_file_path,
                                 includeTypes=includeTypes,
                                 transferDictionaries=True)

# To speed up the computation, we will substitue the IFC objects with their bounding boxes.
boxes = [Topology.BoundingBox(_as_topology(obj)) for obj in ifc_objects]
print(f"Imported {len(ifc_objects)} ifc objects.")

## 8. Show the simplified IFC model

In [ ]:
Topology.Show(boxes,
              width=800,
              height=600,
              backgroundColor="black",
              renderer=renderer)

## 9. Create the spatial relationships graph

In [ ]:
graph = TGraph.BySpatialRelationships(boxes, include=includeRels)
print(f"Created a graph with {len(TGraph.Vertices(graph))} vertices and {len(_edge_records(graph))} edges.")

## 10. Map the edge types and vertex types to colours

In [ ]:
# Edges
# Each edge has a relFwd (relationship forward) and relBwd (relationship backward)
# For example: an edge can have relFwd = "within" and relBwd = "contains" A is within B therefore B contains A
edges = _edge_records(graph)
for e in edges:
    d = _record_dictionary(e)
    relFwd = Dictionary.ValueAtKey(d, "relFwd")
    edgeColor = rels_color_mapping.get(relFwd, "white")
    d = Dictionary.SetValueAtKey(d, "color", edgeColor)
    d = Dictionary.SetValueAtKey(d, "width", 3)
    e = Topology.SetDictionary(e, d)

# Vertices
vertices = TGraph.Vertices(graph)
for v in vertices:
    d = _record_dictionary(v)
    ifc_type = Dictionary.ValueAtKey(d, "IFC_type", "Unknown")
    vertexColor = ifc_color_mapping.get(ifc_type, "white")
    d = Dictionary.SetValuesAtKeys(d, ["size", "color"], [10, vertexColor])
    v = Topology.SetDictionary(v, d)


## 11. Show the final result

In [ ]:
Topology.Show(boxes, graph,
              faceOpacity=0.1,
              sagitta=0.15,
              absolute=False,
              backgroundColor="black",
              vertexSizeKey="size",
              vertexColorKey="color",
              edgeWidthKey="width",
              edgeColorKey="color",
              width=800,
              height=600,
              renderer=renderer)